In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/unsloth_exports/gemma-4-E2B-it-gguf-1")
dst = Path("/content/gemma-4-it")

dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Saved to:", dst)

Saved to: /content/gemma-4-it


In [3]:
!ls /content/gemma-4-it

export_metadata.json  gemma-4-e2b-it.BF16-mmproj.gguf  gemma-4-e2b-it.F16.gguf


In [ ]:
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/unsloth_exports/gemma-4-E2B-run2")
dst = Path("/content/gemma-4-run2")

dst.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(src, dst, dirs_exist_ok=True)

print("Saved to:", dst)

Saved to: /content/gemma-4-run2


In [5]:
!apt update -y && apt upgrade -y
!apt install -y cmake build-essential git

!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp

!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,391 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
107 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire o

In [8]:
!ls /content/llama.cpp/build/bin

export-graph-ops	       llama-perplexity
libggml-base.so		       llama-q8dot
libggml-base.so.0	       llama-quantize
libggml-base.so.0.11.1	       llama-qwen2vl-cli
libggml-cpu.so		       llama-results
libggml-cpu.so.0	       llama-retrieval
libggml-cpu.so.0.11.1	       llama-save-load-state
libggml-cuda.so		       llama-server
libggml-cuda.so.0	       llama-simple
libggml-cuda.so.0.11.1	       llama-simple-chat
libggml.so		       llama-speculative
libggml.so.0		       llama-speculative-simple
libggml.so.0.11.1	       llama-template-analysis
libllama-common.so	       llama-tokenize
libllama-common.so.0	       llama-tts
libllama-common.so.0.0.9097    llama-vdot
libllama.so		       test-alloc
libllama.so.0		       test-arg-parser
libllama.so.0.0.9097	       test-autorelease
libmtmd.so		       test-backend-ops
libmtmd.so.0		       test-backend-sampler
libmtmd.so.0.0.9097	       test-barrier
llama-batched		       test-c
llama-batched-bench	       test-chat
llama-bench		       test-chat-aut

In [14]:
!lsof -i :8080

COMMAND PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
node      7 root   21u  IPv6    622      0t0  TCP *:8080 (LISTEN)
node      7 root   26u  IPv6 173628      0t0  TCP 214f9e31a3b9:8080->172.28.0.1:56098 (ESTABLISHED)
node      7 root   28u  IPv6 628447      0t0  TCP 214f9e31a3b9:8080->172.28.0.1:60166 (ESTABLISHED)


In [9]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("<token>")

public_url = ngrok.connect(9999, "http")
print("vLLM public URL:", public_url)
print("Chat endpoint:", str(public_url) + "/v1/chat/completions")

In [19]:
!/content/llama.cpp/build/bin/llama-server \
  -m /content/gemma-4-it/gemma-4-e2b-it.F16.gguf \
  --alias gemma-local \
  --host 0.0.0.0 \
  --port 9999 \
  -ngl all \
  -c 40960 \
  -np 5 \
  -cb \
  --jinja \
  --reasoning-budget 0 \
  --api-key my-secret-token

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 22563 MiB):
  Device 0: NVIDIA L4, compute capability 8.9, VMM: yes, VRAM: 22563 MiB
build_info: b9097-0b047287f
system_info: n_threads = 6 (n_threads_batch = 6) / 12 | CUDA : ARCHS = 890 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | AVX512_VNNI = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Running without SSL
init: api_keys: ****oken
init: using 11 threads for HTTP server
start: binding port with default address family
main: loading model
srv    load_model: loading model '/content/gemma-4-it/gemma-4-e2b-it.F16.gguf'
common_init_result: fitting params to device memory, for bugs during this step try to reproduce them with -fit off, or provide --verbose logs if the bug only occurs with -fit on
common_params_fit_impl: getting device memory data for initial parameters:
common_memory_breakdown_print: | memory breakdown [MiB] | total    free    